# Fine-Tuning – Austrian Tax Law

**Model:** Qwen2-1.5B (Unsloth / LoRA Fine-Tuning)

---

## How does the model work?

This notebook fine-tunes a small language model (Qwen2-1.5B) using LoRA (Low-Rank Adaptation)
on a custom dataset of Austrian tax law Q&A pairs extracted from WU exam papers and other tax sources (listed below).
After training, the fine-tuned model is used to answer all 643 questions from the test set.

---

## Setup

1. Upload FT_qa_new.csv (training data) and dataset_clean.csv (test set) to Google Drive
2. Adjust the file paths in Cell 4 if needed
3. Run all cells in order
4. Make sure the runtime is set to T4 GPU: Runtime > Change runtime type > T4 GPU

---

## Data

**Training data:** 113 Q&A pairs manually extracted from WU exam papers on Austrian tax law, accountant Q&As and public inforamtion about tax law.
(topics: EStG, KStG, UStG, BAO). Created as a CSV with columns question and answer.

- https://res-wth.at/service/h%C3%A4ufige-fragen
- https://www.salzburg.gv.at/ehrenamt/fragen-und-antworten-steuerrecht
- https://www.studocu.com/de-at/document/universitat-wien/steuerrecht/falle-steuerrecht/57034172
- https://ksw.or.at/berufszugang-berufsanwaerterinnen/fachpruefungen-wtbg-2017/downloads/#c2123

**Test data:** The dataset_clean.csv contains 643 questions on Austrian tax law.
Each question is identified by a unique id and a prompt in German.
The dataset was used exclusively as a test set — it was never used for training.


## Cell 1 - Install
Installs Unsloth and the datasets library. Unsloth enables
efficient 4-bit LoRA fine-tuning on T4 GPU.

In [1]:
%%capture
!pip install unsloth
!pip install datasets

## Cell 2 - Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Cell 3 - Find CSV files in Drive

In [3]:
import os
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/drive/MyDrive/Colab Notebooks/dataset_clean.csv
/content/drive/MyDrive/Colab Notebooks/output_rag_mistral.csv
/content/drive/MyDrive/Colab Notebooks/FT_qa.csv
/content/drive/MyDrive/Colab Notebooks/FT_qa_new.csv
/content/drive/MyDrive/Colab Notebooks/submission_run3.csv


## Cell 4 – Configuration

Central configuration for all paths, model name, and hyperparameters.
All settings are defined here to keep the notebook reproducible without hardcoded values.

In [4]:
TRAIN_CSV_PATH   = "/content/drive/MyDrive/Colab Notebooks/FT_qa_new.csv"
DATASET_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset_clean.csv"
OUTPUT_CSV_PATH  = "/content/drive/MyDrive/Colab Notebooks/submission_run4.csv"
MODEL_OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/qwen_finetun"

MODEL_NAME = "unsloth/Qwen2-1.5B"

MAX_SEQ_LENGTH    = 1040.       # maximum number of tokens per training example
NUM_EPOCHS        = 5           # number of full passes through the training dataset
BATCH_SIZE        = 2           # samples processed per GPU step
GRAD_ACCUMULATION = 4           # simulates a larger batch by accumulating gradients over 4 steps
LEARNING_RATE     = 2e-4        # learning rate: controls how much the model weights change per update step


LORA_R       = 16               # scaling factor applied to the LoRA update
LORA_ALPHA   = 32               # acts as regularization to prevent overfitting on the small training set
LORA_DROPOUT = 0.05

print("Configuration loaded.")

Configuration loaded.


## Cell 5 – Load base model

Loads the Qwen2-1.5B base model with 4-bit quantization via Unsloth.
4-bit quantization reduces VRAM usage so the model fits on a T4 GPU (14.5 GB).

In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print("Model loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

'The read operation timed out' thrown while requesting HEAD https://huggingface.co/unsloth/Qwen2-1.5B-bnb-4bit/resolve/main/generation_config.json
Retrying in 1s [Retry 1/5].


generation_config.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/107 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

Model loaded.


## Cell 6 – Add LoRA adapter

Attaches a LoRA adapter to the frozen base model. Only ~0.28% of all parameters
are trained, which drastically reduces memory usage and training time
while still adapting the model to the target domain.

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],    # attention layers to adapt
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.4 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


## Cell 7 – Load and prepare training data

Loads the custom Q&A dataset from Google Drive, cleans it, and formats each
example as an instruction-tuning prompt: system instruction + question + answer.
This format teaches the model the desired response style.

In [7]:
import pandas as pd
from datasets import Dataset

df_train = pd.read_csv(TRAIN_CSV_PATH)
df_train.columns = df_train.columns.str.strip().str.lower()

if "question" not in df_train.columns or "answer" not in df_train.columns:
    raise ValueError(f"Columns found: {df_train.columns.tolist()}")

df_train["question"] = df_train["question"].str.strip()
df_train["answer"]   = df_train["answer"].str.strip()
df_train = df_train.dropna(subset=["question", "answer"])
df_train = df_train[df_train["question"].str.len() > 5]
df_train["question"] = df_train["question"].apply(lambda x: x.split("\n")[0].strip())

def format_prompt(row):
    return "\n".join([
        "Du bist ein Experte fuer oesterreichisches Steuerrecht.",
        "Beantworte die folgende Frage praezise und korrekt.",
        "Nenne wenn moeglich den relevanten Paragraphen (z.B. § 1 EStG, § 6 UStG).",
        "Antworte auf Deutsch in 2-4 saetzen.",
        "",
        "### Frage:",
        row["question"],
        "",
        "### Antwort:",
        row["answer"],
    ])

df_train["text"] = df_train.apply(format_prompt, axis=1)
train_dataset = Dataset.from_pandas(df_train[["text"]])

print(f"Training samples: {len(train_dataset)}")
print(train_dataset[0]["text"][:300])

Training samples: 113
Du bist ein Experte fuer oesterreichisches Steuerrecht.
Beantworte die folgende Frage praezise und korrekt.
Nenne wenn moeglich den relevanten Paragraphen (z.B. § 1 EStG, § 6 UStG).
Antworte auf Deutsch in 2-4 saetzen.

### Frage:
Was ist der Unterschied zwischen Bilanz und Einnahmen/Ausgabenrechnun


## Cell 8 – Train

Runs the Supervised Fine-Tuning (SFT) via Unsloth's SFTTrainer.
Training takes approximately 2 minutes for 5 epochs on 113 samples on a T4 GPU.

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bf16_supported

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    fp16=not is_bf16_supported(),
    bf16=is_bf16_supported(),
    logging_steps=5,
    save_strategy="epoch",
    optim="adamw_8bit",
    lr_scheduler_type="cosine",       # learning rate decreases following a cosine curve
    warmup_steps=5,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

print("Starting training...")
trainer.train()
print("Training complete.")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/113 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 113 | Num Epochs = 5 | Total steps = 75
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,358,144 of 1,548,072,448 (0.28% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,2.938558
10,2.638460
15,2.171234
20,1.761159
25,1.849848
30,1.693693
35,1.594050
40,1.673773
45,1.527851
50,1.409580


Training complete.


## Cell 9 - Save model

In [9]:
model.save_pretrained(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print(f"Model saved to {MODEL_OUTPUT_DIR}")

Model saved to /content/drive/MyDrive/Colab Notebooks/qwen_finetun


## Cell 10 - Load model for inference

In [10]:
from unsloth import FastLanguageModel

inference_model, inference_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(inference_model)
print("Model ready for inference.")

==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model ready for inference.


## Cell 11 – Inference function

Defines the answer generation function. Uses temperature=0.3 for focused outputs
and repetition_penalty=1.4 to avoid repeated sentences.

In [11]:
def generate_answer(question: str, max_new_tokens: int = 150) -> str:
    prompt = "\n".join([
        "Du bist ein Experte für österreichisches Steuerrecht.",
        "Beantworte die folgende Frage klar und kurz ohne Beispiel.",
        "Antworte mit 2 Sätzen.",
        "",
        "### Frage:",
        question,
        "",
        "### Antwort:",
        "",
    ])
    inputs = inference_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(inference_model.device)

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,            # maximum number of tokens to generate
            do_sample=True,                           # enable sampling (required for temperature)
            temperature=0.3,                          # low temperature: focused, less random output
            top_p=0.9,                                # consider top 90% probability mass
            repetition_penalty=1.4,                   # penalizes repeated tokens to avoid loops
            pad_token_id=inference_tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    answer = inference_tokenizer.decode(generated, skip_special_tokens=True).strip()

    sentences = answer.split(". ")
    seen = []
    for s in sentences:
        if s.strip() not in seen:
            seen.append(s.strip())
    return ". ".join(seen)

## Cell 12 – Run inference on all 643 questions

Generates answers for all 643 questions from dataset_clean.csv.
A checkpoint is saved every 50 answers to Google Drive — if the connection drops,
the process resumes automatically from the last checkpoint.

In [12]:
import torch
import pandas as pd

df_preview = pd.read_csv(DATASET_CSV_PATH)

question_col = None
for col in df_preview.columns:
    if col.lower() in ["question", "frage", "text", "input"]:
        question_col = col
        break
if question_col is None:
    question_col = df_preview.columns[1]

for i in range(5):
    question = str(df_preview[question_col].iloc[i])
    answer = generate_answer(question)
    print(f"--- ID: {df_preview['id'].iloc[i]} ---")
    print(f"Frage: {question}")
    print(f"Antwort: {answer}")
    print()

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

--- ID: CORP-TAX-001 ---
Frage: Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?
Antwort: Die BEMESSUNGSLAGE der KÖRPERSTEUREN liegt auf dem EINTEILIGEN INVESTMENTERGANGSVERLÄUGEN
der Gesellschaft. Die Investitionen sind in zwei Gruppen unterteilt: (1) das gesamtenvertriebene Kapital, welches durch den Verkauf von Aktien oder anderen Erwerbsmitteln an andere Personen ausgegeben wird; sowie (2)
das nichtgesamtverdiente Kapitalet aus allen weiteren Mittel des Unternehmens,
die als Gewinn erzielt werden können. Das kapitaleinländische Kapitel bezieht sich hierbei nur auf das
Kap



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- ID: CORP-TAX-002 ---
Frage: Welche steuerlichen Konsequenzen hat es, wenn eine Körperschaft ein zinsloses Darlehen an ihren Gesellschafter vergibt?
Antwort: Die GESCHWALD-Veranlagung ist auf den gesamten Betriebsbetrieb zu übertragen. Die
Gesellschaft kann sich damit nicht aus dem Verhältnisminderungsverlust profitieren,
da der Eintrag in das Einkommenssteuergeld nach §13 Absatz 4 BStU (EINKOMMENSSTEUGELDE)
nicht erlaubt wird. Der Ausgleichsanteil des betreffenden Geschäftsbetreibers liegt bei einem
zweifelsfreien Vorstellungenstausch zwischen beiden Parteien unter anderem durch einen
Kostenentzug oder einer Gewinnabnahme im



Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- ID: CORP-TAX-003 ---
Frage: Welche Körperschaften sind verpflichtet, sämtliche Einkünfte den Einkünften aus Gewerbebetrieb zuzurechnen?
Antwort: Korporationsgesellschaft (§15 Abs.3 UStG) ist nicht nur von der Verwendung des
Ersatzsteuerschutzes betroffen sondern auch vom Betriebsvermögen abzustreichen,
da es sich um eine gesetzlich erlaubte Finanzierungsmöglichkeit handelt.(Zum Ausgleichs- 
Berechtigungsgesuch: §68a StGB). Der Begriff „Betreibergewinn“ bezieht sich auf das Gesamtbeträge-
gewinne einer Unternehmenskette bzw. eines Geschäftsbürgersystems in einem bestimmten Zeitraums („Tagesg



Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- ID: CORP-TAX-004 ---
Frage: Wie wird eine Dividende einer österreichischen Tochtergesellschaft (a) bei der Tochter und (b) bei der österreichischen Muttergesellschaft steuerlich behandelt?
Antwort: Die toten Gesellschafter haben in Österreich keine Mehrwertsteuern zu bezahlen. Die
Tochteschaften sind nicht mehr als Vorhabensvermögen, sondern werden von den
Mehrkassen abgeleitet: Das ist auch so im Fall eines Verbrauchershauses aufgrund des
Gesetzlichen Vorsatzs zur Erwerbsfinanzierung möglich. Der Wert dieser gesellshäuptliche
Vorhandenseitigkeit kann durch einen Anstieg oder Abfall an Gewinnanteile aus dem
Einkommenssaldo bestimmt werden; es muss aber immer mindestens das Ersparnisvolumen sein,
das sich vom Bet

--- ID: CORP-TAX-005 ---
Frage: Was unterscheidet die steuerliche Behandlung einer offenen von einer verdeckten Ausschüttung auf Ebene der ausschüttenden Körperschaft?
Antwort: Die Versteckte Ausgabe ist eine ausdrücklich oder unvermeidlichen
Ausgaben, bei denen das Einkom

In [14]:
import pandas as pd
from tqdm import tqdm
import os

df_test = pd.read_csv(DATASET_CSV_PATH)

question_col = None
for col in df_test.columns:
    if col.lower() in ["question", "frage", "text", "input"]:
        question_col = col
        break
if question_col is None:
    question_col = df_test.columns[1]

print(f"Using column: '{question_col}'")

CHECKPOINT_PATH = "/content/drive/MyDrive/Colab Notebooks/checkpoint_answers.csv"

# falls bereits ein checkpoint existiert, weitermachen wo aufgehört
if os.path.exists(CHECKPOINT_PATH):
    df_checkpoint = pd.read_csv(CHECKPOINT_PATH)
    answers = df_checkpoint["answer"].tolist()
    start_idx = len(answers)
    print(f"Resuming from index {start_idx}.")
else:
    answers = []
    start_idx = 0
    print("Starting fresh.")

for i, row in tqdm(df_test.iloc[start_idx:].iterrows(), total=len(df_test)-start_idx, desc="Generating answers"):
    answer = generate_answer(str(row[question_col]))
    answers.append(answer)

    # alle 50 antworten speichern
    if len(answers) % 50 == 0:
        df_temp = pd.DataFrame({
            "id": df_test["id"].iloc[:len(answers)].values,
            "answer": answers,
        })
        df_temp.to_csv(CHECKPOINT_PATH, index=False)
        print(f"Checkpoint saved at {len(answers)} answers.")

print(f"Done. {len(answers)} answers generated.")

Using column: 'prompt'
Starting fresh.


Generating answers:   0%|          | 0/643 [00:00<?, ?it/s]Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(D

Checkpoint saved at 50 answers.


Generating answers:  16%|█▌        | 100/643 [10:39<1:07:15,  7.43s/it]Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Checkpoint saved at 100 answers.


...

Checkpoint saved at 550 answers.


Generating answers:  93%|█████████▎| 600/643 [1:02:52<05:26,  7.60s/it]Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Checkpoint saved at 600 answers.


Generating answers: 100%|██████████| 643/643 [1:07:41<00:00,  6.32s/it]

Done. 643 answers generated.


In [13]:
#from tqdm import tqdm

#df_test = pd.read_csv(DATASET_CSV_PATH)
#print(f"Loaded {len(df_test)} questions.")
#print(f"Columns: {df_test.columns.tolist()}")

#question_col = None
#for col in df_test.columns:
 #   if col.lower() in ["question", "frage", "text", "input"]:
  #      question_col = col
   #     break
#if question_col is None:
 #   question_col = df_test.columns[1]

#print(f"Using column: '{question_col}'")

#answers = []
#for i, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Generating answers"):
 #   answer = generate_answer(str(row[question_col]))
  #  answers.append(answer)

#print(f"Done. {len(answers)} answers generated.")

Loaded 643 questions.
Columns: ['id', 'prompt']
Using column: 'prompt'


Generating answers:   0%|          | 0/643 [00:00<?, ?it/s]Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(D

KeyboardInterrupt: 

## Cell 13 - Save submission CSV

In [15]:
df_submission = pd.DataFrame({
    "id": df_test["id"],
    "answer": answers,
})

df_submission.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Saved to: {OUTPUT_CSV_PATH}")
print(df_submission.head(20).to_string())

Saved to: /content/drive/MyDrive/Colab Notebooks/submission_run4.csv
              id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              answer
0   CORP-TAX-001                                             Die steuersichere Einnahmen sind der Grundlagenwert. Die Erwerbsnachweisung\nist das Hauptmittel zur Berechnung des Grundläufens, wobei aber auch andere\nBegriffe wie Gewerbeanstehungen oder Verbrauchswert (Wirtschaftsverkehr) in den\nKostenvorstellungen eingesetzt werden 

## Cell 14 - Sanity check

In [16]:
df_check = pd.read_csv(OUTPUT_CSV_PATH)

assert len(df_check) == 643, f"Expected 643 rows, got {len(df_check)}"
assert "id" in df_check.columns, "Missing 'id' column"
assert "answer" in df_check.columns, "Missing 'answer' column"
assert df_check["answer"].isna().sum() == 0, "Empty answers found"

print("Sanity check passed.")
print(f"Total rows: {len(df_check)}")
print(f"Empty answers: {df_check['answer'].isna().sum()}")
print(f"Avg answer length: {df_check['answer'].str.len().mean():.0f} chars")

Sanity check passed.
Total rows: 643
Empty answers: 0
Avg answer length: 425 chars
